# Practical 03a — A Rule-Based Knowledge System (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 3 |
| **Related lectures** | Lectures 3, 5, 6, 7 and 8 |
| **Duration** | 3 hours |
| **Covers the same work as** | Practical 03 (same concepts, same five tasks, same marks) |
| **Submission** | This notebook, run top to bottom from a clean kernel |

This is an easier route through Practical 03. Nothing is left out — you still build a
knowledge base, two inference engines, an explanation facility and a fault finder — but
every idea is introduced in a small step, most of the code is written for you, and you
fill in the missing pieces. A **CHECK** cell after each exercise tells you whether you got
it right.

## 0. How to use this notebook

1. **Run cells in order**, top to bottom. Click a cell and press **Shift + Enter**.
2. When you see **`___`** (three underscores) in code, that is a **blank** for you to fill in.
   Replace the `___` with your answer, then run the cell again.
3. Run the **CHECK** cell that follows. It prints:
   - `OK` — correct, move on.
   - `NOT YET` — it shows what it expected, what you gave, and a hint.
4. If you see a red error message, the most common cause is a blank you have not filled yet.
   Read the **last line** of the error — it names the problem.
5. If you restart the notebook (Kernel → Restart), run all the cells above the one you are
   on again, starting from the setup cell.

> 💡 **Predict before you run.** Many cells ask *"What do you think this will print?"*
> Say your guess out loud (or write it in a comment) before pressing Shift + Enter. This is
> the fastest way to learn.

## 1. Objectives

By the end of this session you will be able to:

1. Represent facts and rules as data, separate from the code that applies them.
2. Implement forward chaining and derive everything that follows from a set of facts.
3. Implement backward chaining to prove one specific query.
4. Record a proof and use it to answer why and why-not questions.
5. Add assumables and extract conflicts for diagnosis.

## 2. Background — the idea in plain words

**A computer can reach conclusions by following rules, without knowing what any word means.**

Think of the exam office. It has rules such as:

> *A student is **eligible** for the exam **if** they are **registered** **and** their
> **attendance is OK**.*

and it has facts about one student:

> *This student is **registered**. Their **attendance is OK**.*

From these, anyone can conclude *"this student is eligible"*. In this lab you will write a
program that draws such conclusions — and, just as importantly, can **explain** them.

| Word | What it means here | Example |
|---|---|---|
| **atom** | a short name for one statement that is either true or not | `'registered'` |
| **fact** | an atom we are told is true | `'registered'` |
| **rule** | *head* is true **if** every atom in the *body* is true | `('eligible', ['registered', 'attendance_ok'])` |
| **head** | the conclusion of a rule | `'eligible'` |
| **body** | the list of conditions of a rule | `['registered', 'attendance_ok']` |
| **knowledge base** | all the rules and facts together | `RULES` and `FACTS` |
| **inference engine** | the code that applies the rules — written once, for any rules | `forward_chain(...)` |

**Two ways to use the rules (Lecture 5):**

| | Forward chaining | Backward chaining |
|---|---|---|
| Starts from | the facts | one question (a **goal**) |
| Asks | *"What can I conclude from what I know?"* | *"Can I prove this one thing?"* |
| Like | listing every conclusion you can reach | checking a recipe: do I have each ingredient? |
| Result | the set of everything that follows | yes / no, plus the **proof** |

**The big idea of Lecture 3:** the rules are **data** (a Python list), not `if` statements
buried in code. An exam officer could change a rule without a programmer touching the engine.

**Key points to keep in mind while you work:**

- A rule is a head and a body: the head follows if every atom in the body holds.
- Forward chaining always stops; backward chaining can go round in circles on rules that refer back to themselves.
- **"Cannot prove it" is not the same as "it is false".** It may just mean a fact is missing.
- A **conflict** is a set of assumptions that cannot all be true together; a **diagnosis** says which assumption(s) to give up so that every conflict goes away (Task 4 explains this slowly).

## 3. Setup

**Software:** Python 3.10 or later. No packages to install.

Run the cell below **first, every time you open the notebook.** It creates the blank
`___` and a small `check()` helper that the CHECK cells use. You do not need to understand
how `check()` works.

In [14]:
___ = None          # the blank you will replace with your answers

def check(label, got, expected, hint=''):
    """Print OK if got == expected, otherwise NOT YET with a hint."""
    if got == expected:
        print('OK      ', label)
    else:
        print('NOT YET ', label)
        print('          expected:', repr(expected))
        print('          you gave:', repr(got))
        if hint:
            print('          hint    :', hint)

print('Setup done. You are ready to start.')

Setup done. You are ready to start.


## 4. Python warm-up — only what you need today

Five small Python ideas are used in this lab. Each has a demo cell (just run it) and a
tiny exercise. If you already know them, run the cells quickly and move on.

### Warm-up 1 — Tuples, and unpacking them

A **tuple** is a fixed group of values in round brackets. We store each rule as a tuple of
two things: the head and the body. **Unpacking** gives each part its own name in one line.

In [15]:
rule = ('eligible', ['registered', 'attendance_ok'])

print(rule[0])            # the head   (position 0)
print(rule[1])            # the body   (position 1)

head, body = rule         # unpacking: head gets rule[0], body gets rule[1]
print('head =', head)
print('body =', body)

eligible
['registered', 'attendance_ok']
head = eligible
body = ['registered', 'attendance_ok']


**Your turn:** unpack `rule2` into `head2` and `body2`.

In [16]:
rule2 = ('hostel_allowed', ['eligible', 'dues_cleared'])
head2, body2 = rule2
print(head2, '<-', body2)

hostel_allowed <- ['eligible', 'dues_cleared']


In [17]:
# CHECK warm-up 1
check('head2', head2, 'hostel_allowed', 'write: head2, body2 = rule2')
check('body2', body2, ['eligible', 'dues_cleared'], 'write: head2, body2 = rule2')

OK       head2
OK       body2


### Warm-up 2 — Sets: `add` and `in`

A **set** is a collection with **no repeats** and no order. It is perfect for "the things
we know are true". `known.add(x)` puts `x` in; `x in known` asks whether it is there.

In [18]:
known = set(['registered'])     # a set holding one atom
known.add('attendance_ok')
known.add('attendance_ok')      # adding it again changes nothing - no repeats
print(known)
print('registered' in known)    # True
print('eligible' in known)      # False

{'attendance_ok', 'registered'}
True
False


**Your turn:** add `'dues_cleared'` to `known`, then ask whether it is in the set.

In [19]:
known.add("dues_cleared")
is_there = 'dues_cleared' in known
print(known, is_there)

{'attendance_ok', 'registered', 'dues_cleared'} True


In [20]:
# CHECK warm-up 2
check("'dues_cleared' added", 'dues_cleared' in known, True, "known.add('dues_cleared')")
check('is_there', is_there, True, "write: 'dues_cleared' in known")

OK       'dues_cleared' added
OK       is_there


### Warm-up 3 — `all()`: are ALL of these true?

`all(...)` gives `True` only if **every** item is true. We use it to ask *"is every
condition of this rule known?"*:

```python
all(b in known for b in body)      # read it as: "is every b in body also in known?"
```

One surprise: `all([])` — **all of nothing** — is `True`, because there is nothing that fails.

In [21]:
print(all([True, True, True]))     # True
print(all([True, False, True]))    # False - one is not true

known = {'registered', 'attendance_ok'}
body  = ['registered', 'attendance_ok']
print(all(b in known for b in body))                   # every condition known -> True
print(all(b in known for b in ['registered', 'dues_cleared']))   # dues_cleared missing -> False

True
False
True
False


**Your turn — predict.** Write `True` or `False` (no quotes) for each, then run the CHECK.

In [22]:
# a) all(b in known for b in ['registered'])
# b) all(b in known for b in ['eligible', 'registered'])
# c) all(b in known for b in [])            <- careful!
pred_a = True
pred_b = False
pred_c = True

In [23]:
# CHECK warm-up 3
check('a)', pred_a, all(b in known for b in ['registered']), "'registered' is in known")
check('b)', pred_b, all(b in known for b in ['eligible', 'registered']),
      "'eligible' is NOT in known, so not all are true")
check('c)', pred_c, all(b in known for b in []),
      'an empty list has nothing that fails, so all([]) is True')

OK       a)
OK       b)
OK       c)


### Warm-up 4 — `while True`: repeat until nothing changes

Forward chaining repeats the same job **again and again until a whole round changes
nothing**. In Python that is a `while True:` loop that `return`s (or `break`s) when a
round makes no change. Here is the same shape on a toy job: keep halving a number while it
is even.

In [24]:
n = 40
while True:
    changed = False
    if n % 2 == 0:          # even? then halve it
        n = n // 2
        changed = True
        print('halved ->', n)
    if not changed:         # this round changed nothing -> stop
        break
print('stopped at', n)

halved -> 20
halved -> 10
halved -> 5
stopped at 5


**Your turn:** what number does the loop stop at if it starts at `n = 24`?

In [25]:
stops_at = 3

In [26]:
# CHECK warm-up 4
check('24 halved until odd', stops_at, 3, '24 -> 12 -> 6 -> 3, and 3 is odd')

OK       24 halved until odd


### Warm-up 5 — A function that calls itself (recursion)

Backward chaining proves a goal by proving its conditions — using **the same function**.
A function that calls itself is **recursive**. It must have a **stopping case**, or it
calls itself for ever and Python stops it with a `RecursionError`. You will see exactly
that happen in Task 5.

In [27]:
def countdown(n):
    if n == 0:                 # the stopping case
        return 'lift off'
    print(n)
    return countdown(n - 1)    # the function calls itself with a smaller n

print(countdown(3))

def no_stop(n):
    return no_stop(n - 1)      # no stopping case!

try:
    no_stop(3)
except RecursionError:
    print('no_stop never finished -> RecursionError')

3
2
1
lift off
no_stop never finished -> RecursionError


**Your turn:** fill in the stopping case so `count_items` returns how many items a list has.

In [28]:
def count_items(items):
    if items == []:            # stopping case: an empty list has ...
        return 0
    return 1 + count_items(items[1:])    # one item, plus the count of the rest

print(count_items(['a', 'b', 'c']))

3


In [29]:
# CHECK warm-up 5
check('count of 3 items', count_items(['a', 'b', 'c']), 3, 'an empty list has 0 items: return 0')
check('count of 0 items', count_items([]), 0, 'return 0')

OK       count of 3 items
OK       count of 0 items


## 5. Procedure — build the system one piece at a time

Each step below introduces **one** idea. Run the demo, read the explanation, then do the
small exercise and its CHECK.

### Step 1 — Rules and facts as data

First we write down **what each atom means**, in a comment — before any rule (Lecture 4).
Then the rules go in a list, and the facts in another list. Notice there is no `if`
anywhere: the rules are just data.

In [30]:
# MEANINGS - written before any rule, as Lecture 4 insisted
#   registered      : enrolled in this course this semester
#   attendance_ok   : attendance is at least 75 percent
#   dues_cleared    : all fees for this semester are paid
#   eligible        : the student may sit the examination
#   hostel_allowed  : the student may stay in the hostel
#   library_card    : the student may be issued a library card

# A rule is (head, [body atoms]).
RULES = [
    ('eligible',       ['registered', 'attendance_ok']),
    ('hostel_allowed', ['eligible', 'dues_cleared']),
]
FACTS = ['registered', 'attendance_ok', 'dues_cleared']

for head, body in RULES:
    print('%-16s <- %s' % (head, ' AND '.join(body)))
print('facts:', FACTS)

eligible         <- registered AND attendance_ok
hostel_allowed   <- eligible AND dues_cleared
facts: ['registered', 'attendance_ok', 'dues_cleared']


Read `('hostel_allowed', ['eligible', 'dues_cleared'])` as:
*"**hostel_allowed** is true **if** eligible **and** dues_cleared are both true."*

**Exercise 1:** turn this English sentence into a rule and add it to `RULES`:

> *A student may be issued a **library card** if they are **registered** and their **dues are cleared**.*

In [31]:
library_rule = ('library_card', ['registered', 'dues_cleared'])
RULES = RULES[:2] + [library_rule]       # the two rules above, plus yours (safe to re-run)

for head, body in RULES:
    print('%-16s <- %s' % (head, ' AND '.join(body)))

eligible         <- registered AND attendance_ok
hostel_allowed   <- eligible AND dues_cleared
library_card     <- registered AND dues_cleared


In [32]:
# CHECK Step 1
check('the library rule', library_rule, ('library_card', ['registered', 'dues_cleared']),
      "head first, then the list of conditions: ('library_card', ['registered', 'dues_cleared'])")
check('RULES now has 3 rules', len(RULES), 3, 'run the Step 1 cells again, in order')

OK       the library rule
OK       RULES now has 3 rules


### Step 2 — Forward chaining: work out everything that follows

The recipe:

1. Start with `known` = the facts.
2. Go through every rule. If its head is not known yet **and** every atom in its body is
   known, the rule **fires**: add its head to `known`.
3. One trip through all the rules is called a **pass**. Keep doing passes.
4. **Stop when a whole pass adds nothing new.**

Fill in the **three** blanks.

In [33]:
def forward_chain(rules, facts, verbose=True):
    known = set(facts)                              # 1. start with the facts
    pass_number = 0
    while True:
        pass_number = pass_number + 1
        added = False                               # nothing new yet in this pass
        for head, body in rules:                    # 2. look at every rule
            if head not in known and all(b in known for b in body):
                known.add(head)                     #    blank 1: the rule fires - what do we add?
                added = True                        #    blank 2: remember something was added
                if verbose:
                    print('pass %d: derived %-16s from %s' % (pass_number, head, body))
        if not added:                               # 4. a whole pass added nothing
            if verbose:
                print('pass %d: nothing new -> stop' % pass_number)
            return known                            #    blank 3: give back everything we know

derived = forward_chain(RULES, FACTS)
print('\nfinal derived set:', sorted(derived))

pass 1: derived eligible         from ['registered', 'attendance_ok']
pass 1: derived hostel_allowed   from ['eligible', 'dues_cleared']
pass 1: derived library_card     from ['registered', 'dues_cleared']
pass 2: nothing new -> stop

final derived set: ['attendance_ok', 'dues_cleared', 'eligible', 'hostel_allowed', 'library_card', 'registered']


**What happened?** In pass 1, `eligible` was derived first — and because it was now known,
`hostel_allowed` could fire **in the same pass**. Pass 2 found nothing new, so the loop
stopped. The last pass always adds nothing: that is how the program knows it is finished.

**Exercise 2 — predict, then check.** Suppose the student's attendance is **not** OK.
Which atoms will forward chaining end up with? Write the whole set.

In [34]:
FACTS2 = ['registered', 'dues_cleared']            # no attendance_ok this time

# Your prediction - a set, e.g. {'a', 'b'}
predicted = {'registered', 'dues_cleared', 'library_card'}

print('actual:', sorted(forward_chain(RULES, FACTS2, verbose=False)))

actual: ['dues_cleared', 'library_card', 'registered']


In [35]:
# CHECK Step 2
check('full fact set', forward_chain(RULES, FACTS, verbose=False),
      {'registered', 'attendance_ok', 'dues_cleared', 'eligible', 'hostel_allowed', 'library_card'},
      'blank 1 is head, blank 2 is True, blank 3 is known')
check('prediction without attendance_ok', predicted, forward_chain(RULES, FACTS2, verbose=False),
      'no attendance_ok -> not eligible -> no hostel; but library_card needs only registered + dues_cleared')

OK       full fact set
OK       prediction without attendance_ok


### Step 3 — Backward chaining: prove ONE goal

Now we start from a question, e.g. *"Is hostel_allowed true?"* The recipe:

1. If the goal is a **fact** → yes, proved.
2. Otherwise, find a rule whose **head is the goal**, and try to prove **every atom in its
   body** — using this same recipe (that is the recursion from Warm-up 5).
3. If no rule works → it **cannot be proved**.

First, the plain version. Fill in the **three** blanks.

In [36]:
def prove(rules, facts, goal):
    """True if goal can be proved from the facts and rules, otherwise False."""
    if goal in facts:
        return True                                # blank 1: a fact needs no proof
    for head, body in rules:
        if head == goal:                            # blank 2: a rule that concludes our goal
            if all(prove(rules, facts, b) for b in body):    # prove every condition
                return True
    return False                                    # blank 3: no rule worked

print(prove(RULES, FACTS, 'hostel_allowed'))
print(prove(RULES, FACTS2, 'hostel_allowed'))

True
False


In [37]:
# CHECK Step 3 (plain version)
check('hostel_allowed with FACTS', prove(RULES, FACTS, 'hostel_allowed'), True,
      'blank 1 is True, blank 2 is goal')
check('hostel_allowed with FACTS2', prove(RULES, FACTS2, 'hostel_allowed'), False,
      'blank 3 is False: no rule could prove it')
check('a fact', prove(RULES, FACTS, 'registered'), True, 'blank 1: a fact is proved at once')

OK       hostel_allowed with FACTS
OK       hostel_allowed with FACTS2
OK       a fact


**The full version (given — just read and run).** `backward_chain` is the same recipe with
three extras:

| Extra | Why |
|---|---|
| it **prints** each goal, indented by depth | so you can watch the search |
| it **records the proof** in a dictionary `proof` | `proof[atom]` = `'fact'`, or the body of the rule that proved it — this is what makes **explanation** possible (Step 4) |
| it keeps a set `seen` of goals it is already working on | a **loop guard** — if a goal comes back while still being proved, give up instead of going round for ever (Task 5) |

In [38]:
def backward_chain(rules, facts, goal, proof=None, depth=0, seen=None):
    if proof is None:
        proof = {}
    if seen is None:
        seen = set()
    indent = '  ' * depth
    print('%sgoal: %s' % (indent, goal))
    if goal in facts:
        print('%s  -> known fact' % indent)
        proof[goal] = 'fact'
        return True, proof
    if goal in seen:                       # loop guard
        print('%s  -> already being proved: giving up' % indent)
        return False, proof
    seen = seen | {goal}
    for head, body in rules:
        if head == goal:
            print('%s  trying rule %s <- %s' % (indent, head, body))
            if all(backward_chain(rules, facts, b, proof, depth + 2, seen)[0]
                   for b in body):
                proof[goal] = body
                return True, proof
    print('%s  -> cannot establish %s' % (indent, goal))
    return False, proof

ok, proof = backward_chain(RULES, FACTS, 'hostel_allowed')
print('\nproved:', ok)
print('proof :', proof)

goal: hostel_allowed
  trying rule hostel_allowed <- ['eligible', 'dues_cleared']
    goal: eligible
      trying rule eligible <- ['registered', 'attendance_ok']
        goal: registered
          -> known fact
        goal: attendance_ok
          -> known fact
    goal: dues_cleared
      -> known fact

proved: True
proof : {'registered': 'fact', 'attendance_ok': 'fact', 'eligible': ['registered', 'attendance_ok'], 'dues_cleared': 'fact', 'hostel_allowed': ['eligible', 'dues_cleared']}


**Read the trace as a tree.** To prove `hostel_allowed` it needed `eligible` and
`dues_cleared`; to prove `eligible` it needed `registered` and `attendance_ok`, which are facts.

```
hostel_allowed
├── eligible
│   ├── registered       (fact)
│   └── attendance_ok    (fact)
└── dues_cleared         (fact)
```

Notice that `library_card` was **never looked at** — backward chaining only touches the
rules the goal leads it to.

**Exercise 3:** look at the printed `proof` dictionary and copy out two entries.

In [40]:
proof_of_eligible = ['registered', 'attendence_ok']     # what is proof['eligible']?
proof_of_registered = 'fact'                            # what is proof['registered']?

In [41]:
# CHECK Step 3 (proof record)
check("proof['eligible']", proof_of_eligible, proof['eligible'],
      'the body of the rule that proved eligible')
check("proof['registered']", proof_of_registered, proof['registered'],
      "facts are recorded as the string 'fact'")

NOT YET  proof['eligible']
          expected: ['registered', 'attendance_ok']
          you gave: ['registered', 'attendence_ok']
          hint    : the body of the rule that proved eligible
OK       proof['registered']


### Step 4 — Explanation, read straight from the proof

Because the proof was recorded, a **why** question is answered by **reading** the
dictionary — nothing is worked out again.

In [42]:
def why(proof, atom):
    if atom not in proof:
        return '%s was never established.' % atom
    if proof[atom] == 'fact':
        return '%s was given as a fact.' % atom
    return '%s because %s.' % (atom, ' and '.join(proof[atom]))

for atom in ['hostel_allowed', 'eligible', 'registered']:
    print(why(proof, atom))

hostel_allowed because eligible and dues_cleared.
eligible because registered and attendance_ok.
registered was given as a fact.


**Exercise 4:** what does `why(proof, 'library_card')` print? Write the exact sentence.
(Hint: was `library_card` part of the proof of `hostel_allowed`?)

In [45]:
predicted_why = "library_card was never established."

In [44]:
# CHECK Step 4
check('why library_card', predicted_why, why(proof, 'library_card'),
      'backward chaining for hostel_allowed never touched library_card, so it is not in proof')

NOT YET  why library_card
          expected: 'library_card was never established.'
          you gave: None
          hint    : backward chaining for hostel_allowed never touched library_card, so it is not in proof


## 6. Tasks (marked)

These are the same five tasks as Practical 03, worth the same marks. Each one is split
into small parts, with blanks to fill in and a CHECK at the end.

### Task 1 — Model your own domain

> **Marks:** 5

You need a knowledge base of **at least eight rules and five facts**, with the meaning of
every atom written in a comment **before** any rule.

To get you started, we use a **library** domain. The meanings are written for you, and the
rules are given in plain English below. The first four are already turned into code —
**you write rules 5 to 9 and the facts.**

| # | Rule in English |
|---|---|
| 1 | may_borrow **if** member **and** no_dues **and** book_on_shelf *(done)* |
| 2 | may_take_home **if** may_borrow **and** lending_copy *(done)* |
| 3 | extended_loan **if** may_take_home **and** senior_student *(done)* |
| 4 | quiet_room_pass **if** member **and** no_dues *(done)* |
| 5 | may_reserve **if** member **and** no_dues |
| 6 | may_use_lab **if** member **and** lab_inducted |
| 7 | may_borrow_laptop **if** may_use_lab **and** no_dues |
| 8 | overnight_laptop **if** may_borrow_laptop **and** senior_student |
| 9 | may_print **if** may_use_lab **and** printing_quota |

**The student's case (these become the facts):** they are a **member**, owe **no dues**,
the book is **on the shelf**, it is a **lending copy**, and they are a **senior student**.
They have **not** done the lab induction and have **no** printing quota — so those two
are **not** facts.

> **Want to use your own domain instead** (scholarships, a device fault, hostel rules)?
> You may. Keep the same shape — meanings first, at least 8 rules, at least 5 facts — and
> make sure at least one rule **cannot** fire (you need a failing query for Task 3). Later
> cells name library atoms; change those names to yours.

In [46]:
# MEANINGS - every atom, before any rule
#   member            : holds a current library membership for this semester
#   no_dues           : owes no outstanding fine
#   book_on_shelf     : the requested title is physically present in the library
#   lending_copy      : this copy belongs to the lending collection, not the reference one
#   senior_student    : enrolled in year 3 or year 4
#   lab_inducted      : has completed the computing-lab induction
#   printing_quota    : has print credit remaining
#   may_borrow        : permitted to borrow the title at the desk
#   may_take_home     : permitted to remove the copy from the building
#   extended_loan     : entitled to a 28-day loan rather than 14
#   quiet_room_pass   : permitted to book a quiet study room
#   may_reserve       : permitted to place a hold on a title that is out
#   may_use_lab       : permitted to use the computing lab
#   may_borrow_laptop : permitted to borrow a laptop within the building
#   overnight_laptop  : permitted to keep a borrowed laptop overnight
#   may_print         : permitted to send a job to the printers

MY_RULES = [
    ('may_borrow',      ['member', 'no_dues', 'book_on_shelf']),     # rule 1
    ('may_take_home',   ['may_borrow', 'lending_copy']),             # rule 2
    ('extended_loan',   ['may_take_home', 'senior_student']),        # rule 3
    ('quiet_room_pass', ['member', 'no_dues']),                      # rule 4
    ('may_reserve',     ['member', 'no_dues']),                        # rule 5
    ('may_use_lab',     ['member', 'lab_inducted']),                  # rule 6
    ('may_borrow_laptop', ['may_use_lab', 'no_dues']),               # rule 7
    ('overnight_laptop', ['may_borrow_laptop', 'senior_student']),     # rule 8
    ('may_print',       ['may_use_lab', 'printing_quota']),             # rule 9
]
MY_FACTS = ['member', 'no_dues', 'book_on_shelf', 'lending_copy', 'senior_student']

print('%d rules, %d facts' % (len(MY_RULES), len(MY_FACTS)))
for head, body in MY_RULES:
    print('  %-18s <- %s' % (head, ' AND '.join(body)))
print('  facts:', MY_FACTS)

9 rules, 5 facts
  may_borrow         <- member AND no_dues AND book_on_shelf
  may_take_home      <- may_borrow AND lending_copy
  extended_loan      <- may_take_home AND senior_student
  quiet_room_pass    <- member AND no_dues
  may_reserve        <- member AND no_dues
  may_use_lab        <- member AND lab_inducted
  may_borrow_laptop  <- may_use_lab AND no_dues
  overnight_laptop   <- may_borrow_laptop AND senior_student
  may_print          <- may_use_lab AND printing_quota
  facts: ['member', 'no_dues', 'book_on_shelf', 'lending_copy', 'senior_student']


Now run forward chaining on your knowledge base. Which conclusions follow, and which do not?

In [47]:
my_derived = forward_chain(MY_RULES, MY_FACTS, verbose=False)
print('everything that follows:', sorted(my_derived))
print('rule heads NOT derived  :', sorted({h for h, _ in MY_RULES} - my_derived))

everything that follows: ['book_on_shelf', 'extended_loan', 'lending_copy', 'may_borrow', 'may_reserve', 'may_take_home', 'member', 'no_dues', 'quiet_room_pass', 'senior_student']
rule heads NOT derived  : ['may_borrow_laptop', 'may_print', 'may_use_lab', 'overnight_laptop']


In [48]:
# CHECK Task 1
def _task1():
    ok_shape = isinstance(MY_RULES, list) and all(
        isinstance(r, tuple) and len(r) == 2 and isinstance(r[0], str)
        and isinstance(r[1], list) and all(isinstance(b, str) for b in r[1])
        for r in MY_RULES)
    check('every rule looks like (head, [conditions])', ok_shape, True,
          "each rule is ('head', ['a', 'b']) - a string, then a list of strings")
    check('at least 8 rules', len(MY_RULES) >= 8, True, 'fill in rules 5 to 9')
    check('at least 5 facts', isinstance(MY_FACTS, list) and len(MY_FACTS) >= 5, True,
          'MY_FACTS is a list of the atoms that are true for this student')
    if not ok_shape or not isinstance(MY_FACTS, list):
        return
    got = forward_chain(MY_RULES, MY_FACTS, verbose=False)
    heads = {h for h, _ in MY_RULES}
    check('some rule fires', len(heads & got) > 0, True,
          'at least one rule should be able to fire from your facts')
    check('some rule cannot fire (needed in Task 3)', len(heads - got) > 0, True,
          'leave out a fact so that at least one conclusion fails')
_task1()

OK       every rule looks like (head, [conditions])
OK       at least 8 rules
OK       at least 5 facts
OK       some rule fires
OK       some rule cannot fire (needed in Task 3)


### Task 2 — Compare the two procedures

> **Marks:** 4

Which procedure does **less work** for one question? To measure it, we add a **counter**
that goes up by 1 **every time a procedure looks at a rule** — the same rule for counting
in both, otherwise the comparison would be unfair.

Both functions below are copies of the procedures you already have, with a counter added
and the printing removed. Fill in the **two** blanks.

In [49]:
def forward_chain_counted(rules, facts):
    known = set(facts)
    examined = 0
    while True:
        added = False
        for head, body in rules:
            examined = examined + 1                    # blank 1: one more rule looked at
            if head not in known and all(b in known for b in body):
                known.add(head)
                added = True
        if not added:
            return known, examined

def backward_chain_counted(rules, facts, goal, seen=None, counter=None):
    if counter is None:
        counter = {'n': 0}      # a dictionary, so every recursive call shares ONE counter
    if seen is None:
        seen = set()
    if goal in facts:
        return True, counter['n']
    if goal in seen:
        return False, counter['n']
    seen = seen | {goal}
    for head, body in rules:
        counter['n'] = counter['n'] + 1                 # blank 2: one more rule looked at
        if head == goal:
            if all(backward_chain_counted(rules, facts, b, seen, counter)[0] for b in body):
                return True, counter['n']
    return False, counter['n']

Now run both on four questions. (If you used your own domain, put four of your own atoms
in `QUERIES` — two that can be proved and two that cannot.)

In [50]:
QUERIES = ['quiet_room_pass', 'extended_loan', 'may_print', 'overnight_laptop']

print('%-18s %8s %10s %10s' % ('query', 'proved?', 'forward', 'backward'))
results = {}
for goal in QUERIES:
    known, f_count = forward_chain_counted(MY_RULES, MY_FACTS)
    ok, b_count = backward_chain_counted(MY_RULES, MY_FACTS, goal)
    results[goal] = (ok, f_count, b_count)
    print('%-18s %8s %10d %10d' % (goal, ok, f_count, b_count))

query               proved?    forward   backward
quiet_room_pass        True         18          4
extended_loan          True         18          6
may_print             False         18         27
overnight_laptop      False         18         36


**Look at the table and answer** (the answers are words in quotes, or `True`/`False`):

In [51]:
# 1. Did the FORWARD count change from one query to another?  True or False
forward_changes = False

# 2. For the FIRST query in your list, which examined fewer rules: 'forward' or 'backward'?
fewer_first = 'backward'

# 3. For the LAST query in your list, which examined fewer rules: 'forward' or 'backward'?
fewer_last = 'forward'

In [52]:
# CHECK Task 2
def _ref_forward(rules, facts):
    known, n = set(facts), 0
    while True:
        added = False
        for h, body in rules:
            n += 1
            if h not in known and all(b in known for b in body):
                known.add(h); added = True
        if not added:
            return n

def _ref_backward(rules, facts, goal, seen=frozenset(), c=None):
    c = c if c is not None else [0]
    if goal in facts: return True, c[0]
    if goal in seen:  return False, c[0]
    for h, body in rules:
        c[0] += 1
        if h == goal and all(_ref_backward(rules, facts, b, seen | {goal}, c)[0] for b in body):
            return True, c[0]
    return False, c[0]

try:
    check('forward count', forward_chain_counted(MY_RULES, MY_FACTS)[1],
          _ref_forward(MY_RULES, MY_FACTS), 'blank 1: examined + 1')
    for g in QUERIES:
        check('backward count for ' + g, backward_chain_counted(MY_RULES, MY_FACTS, g)[1],
              _ref_backward(MY_RULES, MY_FACTS, g)[1], "blank 2: counter['n'] + 1")
    fwd = {results[g][1] for g in QUERIES}
    check('1. does forward change?', forward_changes, len(fwd) > 1,
          'look at the forward column - is every number the same?')
    def fewer(g):
        ok, f, b = results[g]
        return 'backward' if b < f else 'forward'
    check('2. first query', fewer_first, fewer(QUERIES[0]), 'compare the two numbers on the first row')
    check('3. last query', fewer_last, fewer(QUERIES[-1]), 'compare the two numbers on the last row')
except Exception as err:
    print('NOT YET  an error:', type(err).__name__, '-', err, '(usually a blank still ___)')

OK       forward count
OK       backward count for quiet_room_pass
OK       backward count for extended_loan
OK       backward count for may_print
OK       backward count for overnight_laptop
OK       1. does forward change?
OK       2. first query
OK       3. last query


**Why?** Forward chaining **never sees the question** — it works out *everything*, then
looks the answer up, so its cost is the same for every query. Backward chaining only follows
the rules the question leads to: cheap when the goal is proved early, but **expensive when
the goal fails**, because to be sure something cannot be proved it must try **every** rule
that might have proved it. Copy your numbers into the Observations table (Section 7).

### Task 3 — Answer a why-not question

> **Marks:** 5

When a goal **cannot** be proved, a good system says **what is missing** — not just "no".

For each rule that could have concluded the goal, `why_not` lists the conditions of that
rule that cannot be proved. It uses `provable` (given), which is just Step 3's `prove`
with a loop guard and no printing.

Fill in the **three** blanks.

In [55]:
def provable(rules, facts, goal, seen=frozenset()):
    """Quiet backward chaining: True or False, with a loop guard."""
    if goal in facts:
        return True
    if goal in seen:
        return False
    for head, body in rules:
        if head == goal and all(provable(rules, facts, b, seen | {goal}) for b in body):
            return True
    return False

def why_not(rules, facts, goal):
    """Report which conditions of which rules could not be established.
    Returns a list of (rule, missing_conditions)."""
    if provable(rules, facts, goal):
        print(goal, 'CAN be established - nothing is missing.')
        return []
    report = []
    for head, body in rules:
        if head == goal:                                    # blank 1: a rule that concludes our goal
            missing = []
            for b in body:
                if not provable(rules, facts, b):           # blank 2: this condition cannot be proved
                    missing.append(b)                       # blank 3: remember it
            print('rule  %s <- %s' % (head, ' AND '.join(body)))
            print('      MISSING:', missing)
            report.append(((head, body), missing))
    if not report:
        print(goal, 'is not a fact, and no rule concludes it.')
    return report

FAILING_GOAL = 'may_print'          # own domain? put one of your failing atoms here
print('Why not %s?' % FAILING_GOAL)
answer = why_not(MY_RULES, MY_FACTS, FAILING_GOAL)

Why not may_print?
rule  may_print <- may_use_lab AND printing_quota
      MISSING: ['may_use_lab', 'printing_quota']


`may_print` failed because `may_use_lab` and `printing_quota` are missing. But **why** is
`may_use_lab` missing?
To find the **root cause**, ask *why not* again about each missing atom, and keep going
down. The given `why_not_deep` does exactly that (it calls `why_not` again on each missing
atom — recursion once more).

In [56]:
def why_not_deep(rules, facts, goal, depth=0, seen=frozenset()):
    pad = '    ' * depth
    rules_for_goal = [(h, b) for h, b in rules if h == goal]
    if not rules_for_goal:
        print(pad + '%s: not a fact, and no rule concludes it  <-- ROOT CAUSE' % goal)
        return
    for head, body in rules_for_goal:
        missing = [b for b in body if not provable(rules, facts, b, seen | {goal})]
        print(pad + '%s <- %s   missing: %s' % (head, ' AND '.join(body), missing))
        for m in missing:
            if m not in seen:
                why_not_deep(rules, facts, m, depth + 1, seen | {goal})

for goal in [FAILING_GOAL, 'overnight_laptop']:
    print('=== why not %s? ===' % goal)
    why_not_deep(MY_RULES, MY_FACTS, goal)
    print()

=== why not may_print? ===
may_print <- may_use_lab AND printing_quota   missing: ['may_use_lab', 'printing_quota']
    may_use_lab <- member AND lab_inducted   missing: ['lab_inducted']
        lab_inducted: not a fact, and no rule concludes it  <-- ROOT CAUSE
    printing_quota: not a fact, and no rule concludes it  <-- ROOT CAUSE

=== why not overnight_laptop? ===
overnight_laptop <- may_borrow_laptop AND senior_student   missing: ['may_borrow_laptop']
    may_borrow_laptop <- may_use_lab AND no_dues   missing: ['may_use_lab']
        may_use_lab <- member AND lab_inducted   missing: ['lab_inducted']
            lab_inducted: not a fact, and no rule concludes it  <-- ROOT CAUSE



**Read the output:** which single atom is the root cause of both failures?

In [57]:
ROOT_CAUSE = 'lab_inducted'

In [58]:
# CHECK Task 3
try:
    check('FAILING_GOAL really fails', provable(MY_RULES, MY_FACTS, FAILING_GOAL), False,
          'choose a goal that forward chaining did NOT derive in Task 1')
    expected = []
    for h, body in MY_RULES:
        if h == FAILING_GOAL:
            expected.append(((h, body), [b for b in body if not provable(MY_RULES, MY_FACTS, b)]))
    check('why_not report', answer, expected, 'blank 1 is goal, blank 2 is provable(rules, facts, b), blank 3 is b')
    heads = {h for h, _ in MY_RULES}
    check('root cause is a condition no rule or fact supplies',
          isinstance(ROOT_CAUSE, str) and ROOT_CAUSE not in MY_FACTS and ROOT_CAUSE not in heads
          and any(ROOT_CAUSE in body for _, body in MY_RULES), True,
          'look for the line marked <-- ROOT CAUSE')
except Exception as err:
    print('NOT YET  an error:', type(err).__name__, '-', err, '(usually a blank still ___)')

OK       FAILING_GOAL really fails
OK       why_not report
OK       root cause is a condition no rule or fact supplies


**Important:** the system says `lab_inducted` is missing. That could mean the student
**really has not** done the induction — or that **nobody has typed the record in yet**.
"Not proved" is **not** the same as "false". Keep this in mind for Discussion question 2.

### Task 4 — Diagnosis with assumables

> **Marks:** 5

**The idea, with a car first.** You turn the key and the car does not start. You had
*assumed* two things: **the battery is OK** and **there is fuel**. If both were true, the car
**would** start. It did not — a **contradiction**. So **at least one** assumption is wrong.

| Word | Meaning | Car example |
|---|---|---|
| **assumable** | something we assume is true unless the evidence says otherwise — usually *"this part works"* | `battery_ok`, `fuel_ok` |
| **observation** | what we actually saw | the car did **not** start |
| **`false`** | a special atom meaning *"contradiction"* — the rules derive it when prediction and observation clash | predicted *starts*, saw *does not start* |
| **conflict** | the **smallest** set of assumables that, assumed together, lead to `false` | `{battery_ok, fuel_ok}` |
| **diagnosis** | the **smallest** set of assumables to **give up** so that **every** conflict is broken | `{battery_ok}` *or* `{fuel_ok}` |

The magic: we found a fault **without any rule describing what a broken battery does**.
The rules only describe how things work **correctly**.

**Now in the library.** A student who may take a book home scans it at the desk. If the
**scanner works** and the **network works**, the catalogue records the loan. But the
catalogue **still shows the book on the shelf**. Fill in the blanks of the contradiction rule.

In [59]:
#   ok_scanner           : ASSUMABLE - the barcode scanner at the desk reads correctly
#   ok_network           : ASSUMABLE - the desk's link to the catalogue server is up
#   issue_recorded       : the catalogue shows this copy as issued
#   still_shows_on_shelf : OBSERVED  - the catalogue shows this copy as available
#   false                : a contradiction

ASSUMABLES = ['ok_scanner', 'ok_network']

DIAG_RULES = MY_RULES + [
    # prediction: a permitted borrower at a working desk -> the loan is recorded
    ('issue_recorded', ['may_take_home', 'ok_scanner', 'ok_network']),
    # contradiction: the loan is recorded AND the book still shows on the shelf
    ('false', ['issue_recorded', 'still_shows_on_shelf']),
]
OBSERVATION = ['still_shows_on_shelf']

def derives_false(assumed):
    """Assume these assumables, add the observation, and see if 'false' follows."""
    facts = MY_FACTS + OBSERVATION + list(assumed)
    return 'false' in forward_chain(DIAG_RULES, facts, verbose=False)

**Part A — predict.** For each choice of what we assume, does `false` follow? Write `True`
or `False`. (Hint: `issue_recorded` needs **both** `ok_scanner` **and** `ok_network`.)

In [60]:
PREDICT = {
    'assume nothing':          False,
    'assume only ok_scanner':  False,
    'assume only ok_network':  False,
    'assume both':             True,
}

CASES = {
    'assume nothing':          [],
    'assume only ok_scanner':  ['ok_scanner'],
    'assume only ok_network':  ['ok_network'],
    'assume both':             ['ok_scanner', 'ok_network'],
}
print('%-24s %-10s %-10s' % ('case', 'you said', 'actual'))
for name, assumed in CASES.items():
    print('%-24s %-10s %-10s' % (name, PREDICT[name], derives_false(assumed)))

case                     you said   actual    
assume nothing           False      False     
assume only ok_scanner   False      False     
assume only ok_network   False      False     
assume both              True       True      


**Part B — conflicts and diagnoses (given).** The code below tries **every** group of
assumables, smallest first, and keeps the smallest groups that lead to `false` — the
**conflicts**. Then it finds the smallest groups that contain **at least one member of every
conflict** — the **diagnoses** (withdraw any one of those and no conflict survives).
`combinations(['a', 'b'], 1)` gives `('a',)` and `('b',)`; with `2` it gives `('a', 'b')`.

In [61]:
from itertools import combinations

def find_conflicts(assumables):
    conflicts = []
    for size in range(len(assumables) + 1):              # groups of size 0, 1, 2, ...
        for group in combinations(assumables, size):
            already_covered = any(c <= set(group) for c in conflicts)   # a smaller one already found?
            if derives_false(group) and not already_covered:
                conflicts.append(set(group))
    return conflicts

def find_diagnoses(conflicts, assumables):
    diagnoses = []
    for size in range(len(assumables) + 1):
        for group in combinations(assumables, size):
            hits_every_conflict = all(set(group) & c for c in conflicts)
            already_covered = any(d <= set(group) for d in diagnoses)
            if hits_every_conflict and not already_covered:
                diagnoses.append(set(group))
    return diagnoses

conflicts = find_conflicts(ASSUMABLES)
diagnoses = find_diagnoses(conflicts, ASSUMABLES)
print('conflicts :', [sorted(c) for c in conflicts])
print('diagnoses :', [sorted(d) for d in diagnoses])
for d in diagnoses:
    print('   possible fault:', sorted(d))

conflicts : [['ok_network', 'ok_scanner']]
diagnoses : [['ok_scanner'], ['ok_network']]
   possible fault: ['ok_scanner']
   possible fault: ['ok_network']


**Part C — in your own words (in code).** Copy the results: the conflict as a set, and the
diagnoses as a list of sets.

In [62]:
MY_CONFLICT = {'ok_scanner', 'ok_network'}
MY_DIAGNOSES = [{'ok_scanner'}, {'ok_network'}]

In [63]:
# CHECK Task 4
try:
    check('contradiction rule', DIAG_RULES[-1], ('false', ['issue_recorded', 'still_shows_on_shelf']),
          "('false', ['issue_recorded', 'still_shows_on_shelf'])")
    for name, assumed in CASES.items():
        check('Part A: ' + name, PREDICT[name], derives_false(assumed),
              'false needs issue_recorded, which needs BOTH ok_scanner and ok_network')
    check('Part C: the conflict', [MY_CONFLICT], conflicts, 'the one conflict printed above, as a set')
    _key = lambda s: sorted(s)
    check('Part C: the diagnoses',
          sorted(MY_DIAGNOSES, key=_key) if isinstance(MY_DIAGNOSES, list) else MY_DIAGNOSES,
          sorted(diagnoses, key=_key), 'a list of two one-item sets')
except Exception as err:
    print('NOT YET  an error:', type(err).__name__, '-', err, '(usually a blank still ___)')

OK       contradiction rule
OK       Part A: assume nothing
OK       Part A: assume only ok_scanner
OK       Part A: assume only ok_network
OK       Part A: assume both
OK       Part C: the conflict
OK       Part C: the diagnoses


**What does it mean?** Either the scanner **or** the network is faulty — the evidence cannot
tell which, and the program correctly says so. No rule anywhere says *"a scanner can
break"*; the fault was found purely from how the desk **should** work plus one observation
that disagreed.

### Task 5 — Break it deliberately

> **Marks:** 3

What if a rule (indirectly) refers back to itself? Add these two rules, which go round in
a circle:

> *may_print **if** printer_reserved* and *printer_reserved **if** may_print*

Fill in the blanks.

In [64]:
CYCLIC_RULES = MY_RULES + [
    ('may_print',        ['printer_reserved']),
    ('printer_reserved', ['may_print']),
]

def backward_no_guard(rules, facts, goal):
    """Step 3's prove() - with NO loop guard."""
    if goal in facts:
        return True
    for head, body in rules:
        if head == goal:
            if all(backward_no_guard(rules, facts, b) for b in body):
                return True
    return False

print('1) Backward chaining WITHOUT the loop guard, goal = may_print')
try:
    print('   returned', backward_no_guard(CYCLIC_RULES, MY_FACTS, 'may_print'))
    result_no_guard = 'finished'
except RecursionError:
    print('   RecursionError - it never finished!')
    print('   may_print needs printer_reserved, which needs may_print, which needs ...')
    result_no_guard = 'RecursionError'

print()
print('2) Backward chaining WITH the loop guard (Step 3, unchanged):')
ok, _ = backward_chain(CYCLIC_RULES, MY_FACTS, 'may_print')
print('   returned', ok)

print()
print('3) Forward chaining on the same rules:')
known, examined = forward_chain_counted(CYCLIC_RULES, MY_FACTS)
print('   stopped after examining', examined, 'rules')
print('   may_print derived?       ', 'may_print' in known)
print('   printer_reserved derived?', 'printer_reserved' in known)

1) Backward chaining WITHOUT the loop guard, goal = may_print
   RecursionError - it never finished!
   may_print needs printer_reserved, which needs may_print, which needs ...

2) Backward chaining WITH the loop guard (Step 3, unchanged):
goal: may_print
  trying rule may_print <- ['may_use_lab', 'printing_quota']
    goal: may_use_lab
      trying rule may_use_lab <- ['member', 'lab_inducted']
        goal: member
          -> known fact
        goal: lab_inducted
          -> cannot establish lab_inducted
      -> cannot establish may_use_lab
  trying rule may_print <- ['printer_reserved']
    goal: printer_reserved
      trying rule printer_reserved <- ['may_print']
        goal: may_print
          -> already being proved: giving up
      -> cannot establish printer_reserved
  -> cannot establish may_print
   returned False

3) Forward chaining on the same rules:
   stopped after examining 22 rules
   may_print derived?        False
   printer_reserved derived? False


**Choose the right answer** for each question (write the letter in quotes, e.g. `'a'`).

**Q1. What happened to backward chaining without the loop guard?**
- (a) it returned `True`
- (b) it returned `False`
- (c) it never finished — it kept calling itself until Python stopped it with `RecursionError`

**Q2. Why did forward chaining stop without any problem?**
- (a) it was lucky with the order of the rules
- (b) it only ever **adds** atoms to a set, there are only a limited number of atoms, and it stops as soon as a whole pass adds nothing — so it **must** stop
- (c) it has a hidden loop guard

In [65]:
Q1 = 'c'
Q2 = 'b'

In [66]:
# CHECK Task 5
check('cyclic rules added', CYCLIC_RULES[-2:],
      [('may_print', ['printer_reserved']), ('printer_reserved', ['may_print'])],
      "('may_print', ['printer_reserved']) and ('printer_reserved', ['may_print'])")
check('the run without a guard', result_no_guard, 'RecursionError', 'fill both blanks, then re-run')
check('Q1', Q1, 'c', 'look at what printed under 1)')
check('Q2', Q2, 'b', 'think about Warm-up 4: a loop that stops when a round changes nothing')

OK       cyclic rules added
OK       the run without a guard
OK       Q1
OK       Q2


**Now explain in your own words** (double-click to edit). Two or three sentences.

*Without the loop guard, backward chaining does not finish because it keeps trying to prove `may_print` through `printer_reserved`, which leads back to `may_print` forever. Python eventually stops the recursion with `RecursionError`.*

*Forward chaining does not have this problem because it only adds new atoms to a finite set and stops when a complete pass adds nothing new.*

## 7. Observations

Fill in these tables from **your own** results. Double-click to edit.

**Rules examined (Task 2)**

| Query | Rules examined (forward) | Rules examined (backward) | Result (proved / not proved) |
|-------|---|---|---|
| quiet_room_pass | 18 | 4 | proved |
| extended_loan | 18 | 6 | proved |
| may_print | 18 | 27 | not proved |
| overnight_laptop | 18 | 36 | not proved |  

**Why-not (Task 3)**

| Failing goal | Missing condition(s) | Root cause |
|---|---|---|
| may_print | may_use_lab, printing_quota | lab_inducted |  

**Diagnosis (Task 4)**

| Observation that conflicted | Assumables used | Conflict set | Minimal diagnoses |
|---|---|---|---|
| still_shows_on_shelf | ok_scanner, ok_network | {ok_scanner, ok_network} | {ok_scanner} or {ok_network} |  

**Cyclic rule (Task 5)**

| Procedure | What happened |
|---|---|
| Backward, no loop guard | RecursionError because may_print requires printer_reserved, which requires may_print again. |
| Backward, with loop guard | Returned False because the repeated goal was detected. |
| Forward | Stopped normally after 22 rule examinations; neither cyclic atom was derived. |

## 8. Discussion

Answer in your own words. Two or three sentences each is enough. Sentence starters are
given to help — you may change them.

1. Which procedure examined fewer rules for a single query, and why?
2. Your system failed to prove something. Give two different reasons that could explain the
   failure, only one of which means the statement is false.
3. Your diagnosis found a fault without any rule describing that fault. Explain how, in two
   sentences.

*Your answers:*

1.  Backward chaining examined fewer rules for the proved queries because it follows only the rules needed to establish the requested goal. Forward chaining does not depend on the query, so it examined all 9 rules across 2 passes, for 18 rule examinations every time. For the failed queries, backward chaining examined more rules because it had to search the relevant possibilities before concluding that the goal could not be proved.

2. Reason one: the statement may actually be false because its required conditions are not satisfied. 
Reason two: the required fact may simply be missing from the knowledge base, so the system cannot prove the statement even though it might be true in reality. Therefore, not proved does not automatically mean false.

3. The rules only describe how the library desk should work when the scanner and network are working correctly. When the system predicts an issued loan but the observation still says the book is on the shelf, at least one of the assumptions `ok_scanner` or `ok_network` must be wrong.

## 9. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not
whether you can recite the code.

1. Point to the line where your inference engine mentions a specific rule. (It should not — why does that matter?)
2. Why does forward chaining always terminate?
3. Show me the proof record and use it to explain one conclusion.
4. What is the difference between a conflict and a diagnosis?
5. Your system cannot prove a student is eligible. Does that mean they are ineligible?
6. What broke when you added the cyclic rule, and why was forward chaining unaffected?

## 10. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel (Kernel → Restart & Run All)
- [ ] Every CHECK cell prints `OK`
- [ ] Task 1 has at least eight rules and five facts, with every atom's meaning written first
- [ ] The observations tables are filled in with your actual results
- [ ] The Task 5 explanation and the discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P03a_<your-roll-number>.ipynb`

In [67]:
NAME = "Hamza Rashid"
ROLL_NUMBER = "2025-cse-080"
print(NAME, ROLL_NUMBER)

Hamza Rashid 2025-cse-080
